<a href="https://colab.research.google.com/github/Nxpze/essencial-project1/blob/clinic-interactive-workflow/Clinic_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from datetime import datetime, timedelta, date
import random
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)

In [8]:
# 1. CLASS DEFINITIONS & TRIAGE LOGIC

class Patient:
  def __init__(self, patient_id, fullname, dob, phone_number, allergies="ไม่มี", underlying_disease="ไม่มี",):
    self.patient_id = patient_id
    self.fullname = fullname
    self.dob = dob
    self.phone_number = phone_number
    self.allergies = allergies
    self.underlying_disease = underlying_disease
    self.age = self.age_cal()

# คำนวณอายุจากวันเกิด
  def age_cal(self):
    birthdate = datetime.strptime(self.dob, "%Y-%m-%d").date()
    today = datetime.today().date()
    age = today.year - birthdate.year - ((today.month, today.day) < (birthdate.month, birthdate.day))
    return age

class Queue:
  def __init__(self, queue_id, patient_id, queue_number, symptoms, urgency_level, arrival_time):
    self.queue_id = queue_id
    self.patient_id = patient_id
    self.queue_number = queue_number
    self.symptoms = symptoms
    self.urgency_level = urgency_level  # Red, Yellow, Green, White
    self.arrival_time = arrival_time

# ตั้งค่าเวลารอคอยสูงสุด
    self.waiting_time_threshold = self.get_threshold_by_color()
    self.priority_score = self.get_priority_score()
    self.waiting_time = 0
    self.status = "In Queue"
    self.alert_status = "ปกติ"

# กำหนดเกณฑ์เวลารอคอยสูงสุดตามสีเคส
  def get_threshold_by_color(self):
      thresholds = {
        "Red": 0,  # ตรวจทันที
        "Yellow": 15,  # เร่งด่วน รอได้ไม่เกิน 15 นาที
        "Green": 30,  # ไม่เร่งด่วน รอได้ไม่เกิน 30 นาที
        "White": 45,  # ไม่ฉุกเฉิน รอได้ไม่เกิน 45 นาที
        }
      return thresholds.get(self.urgency_level, 30)

# คะแนนสำหรับ Sort Queue (เรียง Red -> Yellow -> Green -> White)
  def get_priority_score(self):
      scores = {"Red": 1, "Yellow": 2, "Green": 3, "White": 4}
      return scores.get(self.urgency_level, 99)

# ตรวจสอบเวลารอคอยจริงเทียบกับ Arrival Time
  def check_waiting_status(self, current_time):
      elapsed_minutes = (current_time - self.arrival_time).total_seconds() / 60
      self.waiting_time = round(elapsed_minutes, 1)

# เตือนเมื่อรอนานเกิน 30 นาที
      if self.waiting_time >= 30 and self.urgency_level != "Red":
         self.alert_status = "ต้องรีเช็ค"

# รีเช็คโดยพยาบาล หากอาการแย่ลงจะ Fast-track เป็น Red
  def nurse_recheck(self, condition):
      if condition == "อาการแย่ลง":
         self.urgency_level = "Red"
         self.priority_score = 1
         self.alert_status = "ดันคิวทันที (Fast-track)"
         self.waiting_time_threshold = 0
      else:
         self.alert_status = "รีเช็คแล้ว อาการคงเดิม"

# สิ้นสุดการตรวจรักษา -> เปลี่ยนสถานะเป็น Return Case
  def complete_examination(self):
      self.status = "Return Case"


class MedicalRecord:
    DOCTOR = {
        "DOC001": {
          "doctor_name": "นพ. สมชาย ใจดี",
          "specialization": "อายุรกรรม"
       },
        "DOC002": {
          "doctor_name": "พญ. วิภาดา รักษาดี",
          "specialization": "กุมารเวชศาสตร์ (หมอเด็ก)"
       },
        "DOC003": {
         "doctor_name": "นพ. ธนกฤต เก่งกาจ",
         "specialization": "ศัลยกรรมกระดูกและข้อ"
       }
             }
    def __init__(self, record_id, patient_id, doctor_id, record_date, diagnosis, status, prescribed_meds, ):
      self.record_id = record_id
      self.patient_id = patient_id

      doctor_info = self.DOCTOR.get(doctor_id,
       {"doctor_name": "ไม่พบข้อมูลแพทย์",
        "specialization": "ไม่ระบุ"
        })

      self.doctor_id = doctor_id
      self.doctor_name = doctor_info.get("doctor_name")
      self.doctor_specialization = doctor_info.get("specialization")
      self.record_date = record_date
      self.diagnosis = diagnosis
      self.status = status
      self.prescribed_meds = prescribed_meds



class Bill:
  def __init__(self, bill_id, record_id, patient_id, treatment_fee, medication_fee, status):
    self.bill_id = bill_id
    self.record_id = record_id
    self.patient_id = patient_id
    self.treatment_fee = treatment_fee
    self.medication_fee = medication_fee
    self.total_payment = self.calculate_total_payment()
    self.status = status

  def calculate_total_payment(self):
      return round(self.treatment_fee + self.medication_fee, 2)



# 2. HELPER FUNCTIONS FOR TRIAGE & SORTING
# จัดเรียงคิวตาม Priority Score
def sort_queue_by_triage(queue_list):
    return sorted(queue_list,key=lambda q: (q.priority_score, q.arrival_time),)

# 3. WORKFLOW RUNNER (เริ่มคัดกรอง -> จบที่ RETURN CASE)
# จำลองผู้ป่วยเข้ามา 4 ราย

patients_data = [
    ("P001", "กิตติ มีสุข", "1995-05-20", "0811112222", "ไม่มี", "เบาหวาน"),
    ("P002", "นภา มั่นคง", "2018-09-12", "0822223333", "ยาพารา", "ไม่มี"),
    ("P003", "สมชาย เจริญ", "1970-01-01", "0833334444", "ไม่มี", "ความดัน"),
    ("P004", "วิภา รุ่งเรือง", "1988-11-30", "0844445555", "อาหารทะเล", "ไม่มี"),]

patients_dict = {}
for patient_id, name, dob, phone, allergy, disease in patients_data:
    patients_dict[patient_id] = Patient(patient_id, name, dob, phone, allergy, disease)

# สร้างคิวและคัดกรองสี
sample_symptoms = [
    ("P001", "ไอ มีน้ำมูก", "Green"),
    ("P002", "ขอใบรับรองแพทย์", "White"),
    ("P003", "หมดสติ แน่นหน้าอก", "Red"),
    ("P004", "ปวดท้องเกร็ง ไข้สูง", "Yellow"),]

start_time = datetime(2026, 8, 23, 9, 0, 0)
queue_list = []

print("=== 1. คัดกรองผู้ป่วยและสร้างคิว ===")
for i, (patient_id, symptom, color) in enumerate(sample_symptoms, 1):
    arr_time = start_time + timedelta(minutes=i * 5)
    q = Queue(f"Q{i:03d}", patient_id, f"A-{i:03d}", symptom, color, arr_time)
    queue_list.append(q)
    print(f"คิว {q.queue_number} | รหัสผู้ป่วย: {q.patient_id} ({patients_dict[patient_id].fullname}) | สี: {q.urgency_level} | เวลามาถึง: {q.arrival_time.strftime('%H:%M')}")

# จัดเรียงคิวตามระดับความรุนแรง
print("\n=== 2. จัดเรียงลำดับคิวตามความรุนแรง ===")
queue_list = sort_queue_by_triage(queue_list)
for pos, q in enumerate(queue_list, 1):
    print(f"ลำดับที่ {pos}: คิว {q.queue_number} (เคสสี {q.urgency_level})")

# จำลองการเช็คเวลารอคอยและการรีเช็คโดยพยาบาล
print("\n=== 3. ประเมินเวลารอคอย และ Re-evaluation ===")
current_sim_time = start_time + timedelta(minutes=40)

for q in queue_list:
    q.check_waiting_status(current_sim_time)

    # เพิ่มการแสดงผลเวลารอคอยล่าสุดแบบชัดเจน
    print(f"เวลารอคอยล่าสุด ณ เวลา {current_sim_time.strftime('%H:%M')} น. | "
          f"คิว: {q.queue_number} | เวลาที่รอไปแล้ว: {q.waiting_time} นาที | "
          f"สถานะเตือน: {q.alert_status}")

# สุ่มจำลองเคสปวดท้อง (P004) มีอาการแย่ลงเพื่อทดสอบ Fast-track
    if q.alert_status == "ต้องรีเช็ค":
       condition = "อาการแย่ลง" if q.patient_id == "P004" else "อาการคงเดิม"
       q.nurse_recheck(condition)
       print(f"คิว {q.queue_number} ผลรีเช็คพยาบาล: {condition} -> สถานะใหม่: {q.urgency_level} ({q.alert_status})")

# จัดลำดับคิวใหม่อีกครั้งหลังมี Fast-track
queue_list = sort_queue_by_triage(queue_list)

# เข้าห้องตรวจ บันทึก Medical Record
print("\n=== 4. ตรวจรักษาและจบขั้นตอนที่ RETURN CASE ===")
completed_records = []
doctors_keys = ["DOC001", "DOC002", "DOC003"]

# แพทย์ตรวจเสร็จ เปลี่ยนสถานะเป็น Return Case
for q in queue_list:
    q.complete_examination()

# เลือกหมอและบันทึกเวชระเบียน
    doc_id = random.choice(doctors_keys)
    rec = MedicalRecord(
      record_id=f"REC-{q.queue_id}",
      patient_id=q.patient_id,
      doctor_id=doc_id,
      record_date=current_sim_time.strftime("%Y-%m-%d"),
      diagnosis=f"วินิจฉัยเคส {q.urgency_level}: {q.symptoms}",
      status=q.status, prescribed_meds="รับยาตามอาการ",)
    completed_records.append(rec)
    print(f"คิว {q.queue_number} | แพทย์ผู้ตรวจ: {rec.doctor_name} ({rec.doctor_specialization}) | สถานะเคส: {q.status}")

# 5. พบแพทย์ / วินิจฉัย
# นำข้อมูลจาก Medical Record มาใช้ในการวินิจฉัยผู้ป่วย

print("\n=== 5. พบแพทย์ / วินิจฉัย ===")

treatment_results = []

for rec in completed_records:

    # สั่งยา / นัดหัตถการ
    treatment = rec.prescribed_meds

    # ผลการรักษา
    result = random.choice(["หายขาด", "ไม่หายขาด"])

    # ตรวจสอบว่าผู้ป่วยหายขาดหรือไม่
    if result == "หายขาด":
        appointment = "ไม่ต้องนัด"
    else:
        appointment = "return ใบนัด"

    # เก็บข้อมูลผลการรักษา
    treatment_results.append({
        "record_id": rec.record_id,
        "patient_id": rec.patient_id,
        "treatment": treatment,
        "treatment_result": result,
        "appointment": appointment
    })

    print(
        f"ผู้ป่วย: {rec.patient_id} | "
        f"สั่งยา/นัดหัตถการ: {treatment} | "
        f"ผลการรักษา: {result} | "
        f"{appointment}"
    )


# 6. Bills
# สร้าง Bill หลังจากจบขั้นตอนการรักษา

print("\n=== 6. Bills ===")

completed_bills = []

for i, rec in enumerate(completed_records, 1):

    # กำหนดค่ารักษาและค่ายา
    treatment_fee = random.choice([300, 500, 800, 1000, 1500])
    medication_fee = random.choice([50, 100, 200, 300, 500])

    # สร้าง Bill ของผู้ป่วย
    bill = Bill(
        bill_id=f"BILL-{i:03d}",
        record_id=rec.record_id,
        patient_id=rec.patient_id,
        treatment_fee=treatment_fee,
        medication_fee=medication_fee,
        status="รอชำระ"
    )

    # เก็บข้อมูล Bill
    completed_bills.append(bill)

    print(
        f"ผู้ป่วย: {bill.patient_id} | "
        f"ค่ารักษา: {bill.treatment_fee} บาท | "
        f"ค่ายา: {bill.medication_fee} บาท | "
        f"รวม: {bill.total_payment} บาท"
    )

=== 1. คัดกรองผู้ป่วยและสร้างคิว ===
คิว A-001 | รหัสผู้ป่วย: P001 (กิตติ มีสุข) | สี: Green | เวลามาถึง: 09:05
คิว A-002 | รหัสผู้ป่วย: P002 (นภา มั่นคง) | สี: White | เวลามาถึง: 09:10
คิว A-003 | รหัสผู้ป่วย: P003 (สมชาย เจริญ) | สี: Red | เวลามาถึง: 09:15
คิว A-004 | รหัสผู้ป่วย: P004 (วิภา รุ่งเรือง) | สี: Yellow | เวลามาถึง: 09:20

=== 2. จัดเรียงลำดับคิวตามความรุนแรง ===
ลำดับที่ 1: คิว A-003 (เคสสี Red)
ลำดับที่ 2: คิว A-004 (เคสสี Yellow)
ลำดับที่ 3: คิว A-001 (เคสสี Green)
ลำดับที่ 4: คิว A-002 (เคสสี White)

=== 3. ประเมินเวลารอคอย และ Re-evaluation ===
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-003 | เวลาที่รอไปแล้ว: 25.0 นาที | สถานะเตือน: ปกติ
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-004 | เวลาที่รอไปแล้ว: 20.0 นาที | สถานะเตือน: ปกติ
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-001 | เวลาที่รอไปแล้ว: 35.0 นาที | สถานะเตือน: ต้องรีเช็ค
คิว A-001 ผลรีเช็คพยาบาล: อาการคงเดิม -> สถานะใหม่: Green (รีเช็คแล้ว อาการคงเดิม)
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-002 | เวลาที่รอไปแล้ว

# 4. Interactive Clinic Workflow (ส่วนที่เพิ่มต่อจากโค้ดเดิม)

ส่วนนี้ **ใช้คลาส `Patient`, `Queue`, `MedicalRecord`, `Bill` และข้อมูลจำลอง `P001–P004` ของไฟล์เดิมโดยตรง** แล้วเพิ่มช่องรับข้อมูลด้วย `input()` ตามตัวอย่าง `demo.ipynb`

ลำดับการทำงาน: ค้นหา/ลงทะเบียนผู้ป่วย → กรอกอาการ → คัดกรอง Red/Yellow/Green/White → จัดคิวตามสีและเวลามาถึง → ตรวจเวลารอ/รีเช็ค → พบแพทย์ → ผลการรักษา/ใบนัด → Bill


In [9]:
# 4.1 HELPER FUNCTIONS & INTERACTIVE WORKFLOW
# ส่วนนี้เพิ่มต่อจากโค้ดเดิม โดยไม่แก้คลาสและข้อมูลจำลองด้านบน

import re

if "appointment_records" not in globals():
    appointment_records = []


def prompt_nonempty(message):
    """รับข้อความที่ห้ามเว้นว่าง"""
    while True:
        value = input(message).strip()
        if value:
            return value
        print("❌ ช่องนี้ห้ามเว้นว่าง กรุณากรอกอีกครั้ง")


def prompt_yes_no(message, default=None):
    """รับคำตอบ Y/N โดยคืนค่า True/False"""
    yes_values = {"y", "yes", "1", "true", "ใช่", "ช", "ค่ะ", "ครับ"}
    no_values = {"n", "no", "0", "false", "ไม่", "ม"}
    suffix = " [Y/N]"
    if default is True:
        suffix = " [Y/n]"
    elif default is False:
        suffix = " [y/N]"

    while True:
        answer = input(message + suffix + ": ").strip().lower()
        if not answer and default is not None:
            return default
        if answer in yes_values:
            return True
        if answer in no_values:
            return False
        print("❌ กรุณาตอบ Y หรือ N")


def prompt_choice(message, choices, default=None):
    """รับค่าจาก choices แบบไม่สนตัวพิมพ์เล็ก/ใหญ่"""
    lookup = {choice.lower(): choice for choice in choices}
    choice_text = "/".join(choices)
    while True:
        suffix = f" [{choice_text}]"
        if default:
            suffix += f" (ค่าเริ่มต้น {default})"
        answer = input(message + suffix + ": ").strip()
        if not answer and default:
            return default
        selected = lookup.get(answer.lower())
        if selected:
            return selected
        print(f"❌ กรุณาเลือก {choice_text}")


def prompt_birthdate(message="วันเกิด (YYYY-MM-DD หรือ YYYY/MM/DD): "):
    """ตรวจรูปแบบวันเกิดก่อนส่งให้ Patient class เดิม"""
    while True:
        value = input(message).strip()
        birthdate = None
        for date_format in ("%Y-%m-%d", "%Y/%m/%d"):
            try:
                birthdate = datetime.strptime(value, date_format).date()
                break
            except ValueError:
                continue
        if birthdate and birthdate <= datetime.today().date():
            # Patient class เดิมรับ YYYY-MM-DD จึงแปลงให้อัตโนมัติ
            return birthdate.strftime("%Y-%m-%d")
        print(
            "❌ วันเกิดไม่ถูกต้อง กรุณาใช้ YYYY-MM-DD หรือ YYYY/MM/DD "
            "และต้องไม่เป็นอนาคต"
        )


def prompt_phone(message="เบอร์โทรศัพท์: "):
    while True:
        phone = input(message).strip()
        digits = re.sub(r"\D", "", phone)
        if 9 <= len(digits) <= 15:
            return phone
        print("❌ เบอร์โทรศัพท์ต้องมีตัวเลข 9–15 หลัก")


def prompt_money(message, default=0.0):
    while True:
        value = input(f"{message} [ค่าเริ่มต้น {default:,.2f}]: ").strip()
        try:
            amount = float(value) if value else float(default)
            if amount < 0:
                raise ValueError
            return amount
        except ValueError:
            print("❌ กรุณากรอกจำนวนเงินตั้งแต่ 0 ขึ้นไป")


def next_id(prefix, existing_ids, width=3, separator=""):
    """สร้างรหัสถัดไปจากรหัสที่มีอยู่ โดยไม่ทับข้อมูลเดิม"""
    numbers = []
    pattern = re.compile(rf"^{re.escape(prefix)}{re.escape(separator)}(\d+)$")
    for item_id in existing_ids:
        match = pattern.match(str(item_id))
        if match:
            numbers.append(int(match.group(1)))
    number = max(numbers, default=0) + 1
    return f"{prefix}{separator}{number:0{width}d}"


def show_patient(patient):
    adult_status = "บรรลุนิติภาวะ" if patient.age >= 20 else "ยังไม่บรรลุนิติภาวะ"
    print(f"รหัสผู้ป่วย: {patient.patient_id}")
    print(f"ชื่อ-นามสกุล: {patient.fullname}")
    print(f"วันเกิด: {patient.dob} | อายุ: {patient.age} ปี ({adult_status})")
    print(f"เบอร์โทร: {patient.phone_number}")
    print(f"แพ้ยา: {patient.allergies} | โรคประจำตัว: {patient.underlying_disease}")


def find_or_register_patient():
    """Flowchart: ตรวจ P_id; ถ้าไม่พบให้ลงทะเบียนและคำนวณอายุ"""
    print("\n--- 1) ค้นหาประวัติ / ลงทะเบียนผู้ป่วย ---")
    requested_id = input(
        "กรอกรหัสผู้ป่วย P_id (ทดลอง P001–P004 หรือกด Enter สำหรับผู้ป่วยใหม่): "
    ).strip().upper()

    if requested_id and requested_id in patients_dict:
        patient = patients_dict[requested_id]
        print("\n✅ พบผู้ป่วยเดิมในระบบ")
        show_patient(patient)
        return patient

    if requested_id:
        if not re.fullmatch(r"[A-Z0-9_-]+", requested_id):
            print("⚠️ รูปแบบ P_id ไม่ถูกต้อง ระบบจะสร้างรหัสใหม่ให้")
            requested_id = ""
        else:
            print(f"\nไม่พบ {requested_id} — เข้าสู่การลงทะเบียนผู้ป่วยใหม่")
    else:
        print("\nเข้าสู่การลงทะเบียนผู้ป่วยใหม่")

    patient_id = requested_id or next_id("P", patients_dict.keys())
    fullname = prompt_nonempty("ชื่อ-นามสกุล: ")
    dob = prompt_birthdate()
    phone = prompt_phone()
    allergies = input("ประวัติแพ้ยา (กด Enter หากไม่มี): ").strip() or "ไม่มี"
    disease = input("โรคประจำตัว (กด Enter หากไม่มี): ").strip() or "ไม่มี"

    patient = Patient(patient_id, fullname, dob, phone, allergies, disease)
    patients_dict[patient_id] = patient
    print("\n✅ ลงทะเบียนสำเร็จ")
    show_patient(patient)
    return patient


def triage_from_flowchart():
    """คัดกรองจากระดับรุนแรงที่สุดลงมาตาม flowchart"""
    print("\n--- 2) คัดกรองความเร่งด่วนตาม Flowchart ---")
    if prompt_yes_no("หัวใจหยุดเต้น ไม่หายใจ ช็อก หรือหมดสติหรือไม่", default=False):
        return "Red"
    if prompt_yes_no(
        "หายใจลำบากปานกลาง เจ็บแน่นหน้าอกมาก หรือปวดท้องรุนแรงหรือไม่",
        default=False,
    ):
        return "Yellow"
    if prompt_yes_no(
        "ไข้หวัด ปวดศีรษะเล็กน้อย หรือมีแผลหรือไม่",
        default=False,
    ):
        return "Green"
    if prompt_yes_no(
        "ขอรับยาต่อเนื่อง หรืออื่น ๆ หรือไม่",
        default=False,
    ):
        return "White"

    print("⚠️ Case ไม่สามารถระบุได้ กรุณาให้ผู้มีประสบการณ์ตัดสินใจ")
    return prompt_choice("เจ้าหน้าที่ประเมินเป็นเคสสีใด", ["Red", "Yellow", "Green", "White"])


def create_interactive_queue(patient, symptoms, urgency_level):
    queue_id = next_id("Q", [queue.queue_id for queue in queue_list])
    queue_number = next_id(
        "A", [queue.queue_number for queue in queue_list], separator="-"
    )
    queue = Queue(
        queue_id,
        patient.patient_id,
        queue_number,
        symptoms,
        urgency_level,
        datetime.now(),
    )
    queue_list.append(queue)
    queue_list[:] = sort_queue_by_triage(queue_list)
    return queue


def show_active_queue(current_time=None):
    """อัปเดตเวลารอและแสดงเฉพาะคิวที่ยังรอตรวจ"""
    current_time = current_time or datetime.now()
    active = [queue for queue in queue_list if queue.status == "In Queue"]
    for queue in active:
        queue.check_waiting_status(current_time)
    active = sort_queue_by_triage(active)

    print("\nลำดับคิวที่กำลังรอตรวจ")
    if not active:
        print("- ไม่มีคิวรอตรวจ")
    for position, queue in enumerate(active, 1):
        print(
            f"{position}. {queue.queue_number} | ผู้ป่วย {queue.patient_id} | "
            f"สี {queue.urgency_level} | รอ {queue.waiting_time:.1f} นาที | {queue.alert_status}"
        )
    return active


def recheck_if_over_threshold(queue, current_time=None):
    """Flowchart: ถ้ารอเกิน threshold ต้องรีเช็คและ sort queue ใหม่"""
    current_time = current_time or datetime.now()
    queue.check_waiting_status(current_time)

    if queue.urgency_level == "Red":
        print("🚨 เคส Red ต้องเข้าตรวจทันที")
        return queue

    if queue.waiting_time <= queue.waiting_time_threshold:
        print(
            f"✅ เวลารอ {queue.waiting_time:.1f} นาที ยังไม่เกินเกณฑ์ "
            f"{queue.waiting_time_threshold} นาที"
        )
        return queue

    print(
        f"⚠️ รอ {queue.waiting_time:.1f} นาที เกินเกณฑ์ "
        f"{queue.waiting_time_threshold} นาที"
    )
    print("🔁 เรียกประเมินอาการซ้ำ อัปเดต Case และ Sort Queue ใหม่")
    updated_symptoms = input(
        f"อาการล่าสุด [กด Enter เพื่อใช้ข้อมูลเดิม: {queue.symptoms}]: "
    ).strip()
    if updated_symptoms:
        queue.symptoms = updated_symptoms

    # วนกลับไปตรวจเงื่อนไข Red → Yellow → Green → White ตาม flowchart อีกครั้ง
    queue.urgency_level = triage_from_flowchart()
    queue.priority_score = queue.get_priority_score()
    queue.waiting_time_threshold = queue.get_threshold_by_color()
    queue.alert_status = "รีเช็คแล้ว อัปเดต Case และจัดคิวใหม่"
    # เริ่มนับรอบเฝ้าระวังใหม่หลังประเมินซ้ำ
    queue.arrival_time = current_time
    queue.waiting_time = 0
    queue_list[:] = sort_queue_by_triage(queue_list)
    print(f"✅ Case หลังประเมินซ้ำ: {queue.urgency_level} | {queue.alert_status}")
    return queue


def prompt_appointment_date():
    default_date = (datetime.today().date() + timedelta(days=7)).strftime("%Y-%m-%d")
    while True:
        value = input(
            f"วันที่นัด (YYYY-MM-DD หรือ YYYY/MM/DD) "
            f"[ค่าเริ่มต้น {default_date}]: "
        ).strip()
        value = value or default_date
        appointment_date = None
        for date_format in ("%Y-%m-%d", "%Y/%m/%d"):
            try:
                appointment_date = datetime.strptime(value, date_format).date()
                break
            except ValueError:
                continue
        if appointment_date and appointment_date >= datetime.today().date():
            return appointment_date.strftime("%Y-%m-%d")
        print(
            "❌ วันนัดต้องเป็นวันนี้หรืออนาคต และใช้รูปแบบ "
            "YYYY-MM-DD หรือ YYYY/MM/DD"
        )


def run_clinic_workflow():
    """รับผู้ป่วยหนึ่งรายตั้งแต่เริ่มต้นจนออก Bill ตาม flowchart"""
    print("\n" + "=" * 68)
    print("🏥 ระบบบริการผู้ป่วยตาม Clinic Flowchart")
    print("=" * 68)

    # 1. Patient
    patient = find_or_register_patient()

    # 2. Symptoms & Triage
    symptoms = prompt_nonempty("\nกรอกอาการสำคัญ: ")
    urgency_level = triage_from_flowchart()
    registered_queue = create_interactive_queue(patient, symptoms, urgency_level)
    print(
        f"\n🎫 ออกคิว {registered_queue.queue_number} สำเร็จ | "
        f"สี {registered_queue.urgency_level} | "
        f"เกณฑ์รอสูงสุด {registered_queue.waiting_time_threshold} นาที"
    )

    # 3. Sort Queue & Waiting-time Re-evaluation
    while True:
        active_queues = show_active_queue()
        if not active_queues:
            raise RuntimeError("ไม่พบคิวที่กำลังรอตรวจ")
        queue = active_queues[0]
        queue.check_waiting_status(datetime.now())
        is_over_threshold = (
            queue.urgency_level != "Red"
            and queue.waiting_time > queue.waiting_time_threshold
        )
        if is_over_threshold:
            recheck_if_over_threshold(queue)
            continue
        break

    # เรียกคิวแรกหลัง Sort Case → Sort Queue ตามสีและ arrival_time
    patient = patients_dict[queue.patient_id]
    print(
        f"\n📣 เรียกคิวตามลำดับ: {queue.queue_number} | "
        f"สี {queue.urgency_level} | ผู้ป่วย {patient.fullname}"
    )
    input("\nกด Enter เมื่อพร้อมส่งผู้ป่วยเข้าห้องตรวจ...")

    # 4. Doctor Examination & Medical Record
    print("\n--- 3) พบแพทย์ / วินิจฉัย ---")
    for doctor_id, doctor_info in MedicalRecord.DOCTOR.items():
        print(
            f"{doctor_id}: {doctor_info['doctor_name']} "
            f"({doctor_info['specialization']})"
        )
    doctor_id = prompt_choice(
        "เลือกแพทย์",
        list(MedicalRecord.DOCTOR.keys()),
        default="DOC001",
    )
    diagnosis = prompt_nonempty("ผลการวินิจฉัย: ")
    prescribed_meds = input("สั่งยา / นัดหัตถการ (กด Enter หากไม่มี): ").strip()
    prescribed_meds = prescribed_meds or "ไม่มีการสั่งยา / นัดหัตถการ"

    queue.complete_examination()
    record_id = f"REC-{queue.queue_id}"
    record = MedicalRecord(
        record_id,
        patient.patient_id,
        doctor_id,
        datetime.today().strftime("%Y-%m-%d"),
        diagnosis,
        queue.status,
        prescribed_meds,
    )
    completed_records.append(record)

    # 5. Treatment Result & Appointment
    print("\n--- 4) ผลการรักษา ---")
    recovered = prompt_yes_no("ผู้ป่วยหายขาดหรือไม่", default=True)
    appointment = None
    if recovered:
        appointment_text = "ไม่ต้องนัด"
    else:
        print("\n--- 5) Return ใบนัด ---")
        appointment = {
            "appointment_id": next_id(
                "APT", [item["appointment_id"] for item in appointment_records]
            ),
            "patient_id": patient.patient_id,
            "fullname": patient.fullname,
            "doctor_id": doctor_id,
            "doctor_name": record.doctor_name,
            "phone_number": patient.phone_number,
            "appointment_date": prompt_appointment_date(),
            "reason": input("เหตุผลการนัด [ค่าเริ่มต้น: ติดตามผลการรักษา]: ").strip()
            or "ติดตามผลการรักษา",
        }
        appointment_records.append(appointment)
        appointment_text = (
            f"{appointment['appointment_id']} วันที่ {appointment['appointment_date']} "
            f"({appointment['reason']})"
        )

    treatment_results.append({
        "record_id": record.record_id,
        "patient_id": patient.patient_id,
        "treatment": record.prescribed_meds,
        "treatment_result": "หายขาด" if recovered else "ไม่หายขาด",
        "appointment": appointment_text,
    })

    # 6. Bill
    print("\n--- 6) Bills ---")
    treatment_fee = prompt_money("ค่ารักษา/หัตถการ", default=300.0)
    medication_fee = prompt_money("ค่ายา/เวชภัณฑ์", default=0.0)
    paid = prompt_yes_no("ชำระเงินแล้วหรือไม่", default=False)
    bill_id = next_id(
        "BILL", [bill.bill_id for bill in completed_bills], separator="-"
    )
    bill = Bill(
        bill_id,
        record.record_id,
        patient.patient_id,
        treatment_fee,
        medication_fee,
        "ชำระเงินแล้ว" if paid else "รอชำระ",
    )
    completed_bills.append(bill)

    print("\n" + "=" * 68)
    print("📋 สรุปการรับบริการ")
    print("=" * 68)
    show_patient(patient)
    print(f"คิว: {queue.queue_number} | สี: {queue.urgency_level}")
    print(f"แพทย์: {record.doctor_name} ({record.doctor_specialization})")
    print(f"วินิจฉัย: {record.diagnosis}")
    print(f"ยา/หัตถการ: {record.prescribed_meds}")
    print(f"ใบนัด: {appointment_text}")
    print(f"เลขที่ Bill: {bill.bill_id}")
    print(f"ค่ารักษา: {bill.treatment_fee:,.2f} บาท")
    print(f"ค่ายา: {bill.medication_fee:,.2f} บาท")
    print(f"รวม: {bill.total_payment:,.2f} บาท | สถานะ: {bill.status}")
    print("=" * 68)
    return {
        "patient": patient,
        "registered_queue": registered_queue,
        "queue": queue,
        "record": record,
        "appointment": appointment,
        "bill": bill,
    }


## 5. เปิดระบบรับข้อมูล

รันเซลล์ด้านล่างแล้ว Colab จะแสดงช่องให้กรอกทีละขั้นตอน หากต้องการรับผู้ป่วยรายถัดไปให้รันเซลล์นี้อีกครั้ง ข้อมูลในรอบ runtime เดียวกันจะยังอยู่


In [10]:

clinic_result = run_clinic_workflow()



🏥 ระบบบริการผู้ป่วยตาม Clinic Flowchart

--- 1) ค้นหาประวัติ / ลงทะเบียนผู้ป่วย ---
กรอกรหัสผู้ป่วย P_id (ทดลอง P001–P004 หรือกด Enter สำหรับผู้ป่วยใหม่): 

เข้าสู่การลงทะเบียนผู้ป่วยใหม่
ชื่อ-นามสกุล: GFG G
วันเกิด (YYYY-MM-DD หรือ YYYY/MM/DD): 1996/05/22
เบอร์โทรศัพท์: 1231321312
ประวัติแพ้ยา (กด Enter หากไม่มี): 
โรคประจำตัว (กด Enter หากไม่มี): 

✅ ลงทะเบียนสำเร็จ
รหัสผู้ป่วย: P005
ชื่อ-นามสกุล: GFG G
วันเกิด: 1996-05-22 | อายุ: 30 ปี (บรรลุนิติภาวะ)
เบอร์โทร: 1231321312
แพ้ยา: ไม่มี | โรคประจำตัว: ไม่มี

กรอกอาการสำคัญ: GG

--- 2) คัดกรองความเร่งด่วนตาม Flowchart ---
หัวใจหยุดเต้น ไม่หายใจ ช็อก หรือหมดสติหรือไม่ [y/N]: y

🎫 ออกคิว A-005 สำเร็จ | สี Red | เกณฑ์รอสูงสุด 0 นาที

ลำดับคิวที่กำลังรอตรวจ
1. A-005 | ผู้ป่วย P005 | สี Red | รอ 0.0 นาที | ปกติ

📣 เรียกคิวตามลำดับ: A-005 | สี Red | ผู้ป่วย GFG G

กด Enter เมื่อพร้อมส่งผู้ป่วยเข้าห้องตรวจ...

--- 3) พบแพทย์ / วินิจฉัย ---
DOC001: นพ. สมชาย ใจดี (อายุรกรรม)
DOC002: พญ. วิภาดา รักษาดี (กุมารเวชศาสตร์ (หมอเด็ก))
DOC003: นพ. ธน